# pane

> What the main agent and each sub-agent are doing right now, written to a file and drawn in a tmux split beside the chat.

In [ ]:
#| default_exp pane

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| hide
from fastcore.test import test_eq

In [ ]:
#| export
from __future__ import annotations

import json, os, time
from pathlib import Path

from fastcore.script import call_parse
from rich.console import Console, Group
from rich.live import Live
from rich.text import Text

from ramabana.agent import DELEGATE_TOOLS
from shalya.core import one_line as _1

## The snapshot

`now_snapshot` is one plain dict, so the CLI can write it and another process can draw it. `root` is the foreground turn: its step count, how long it has run, and the call it is waiting on. `subs` holds one row per sub-agent run the turn started. A finished sub-agent keeps its row, marked completed, failed or cancelled, until the next turn replaces the root run.

The counting is the CLI's. A root step is a call whose `parent_action_id` names no call on the activity, as in `Ui.working`. A call hangs off a sub-agent through `parent_action_id`, which names the delegate call that spawned it. An `Act` does not record its run, so the delegate call is linked to its run by the question the run was asked, and a sub-agent's calls are counted only when that call started exactly one run. `delegate_search` with several questions runs them on a thread pool, where the per-thread delegate stack is empty, so their calls carry no parent and count as root steps, which is how the CLI counts them too. A read-only sub-agent, the default, runs on the unrecorded catalog: its calls never reach the activity, so its row has a clock and a state but no steps and no current call.

In [ ]:
#| export
def _secs(r, now): return round(((r.ended or now) - r.started) if r.started else 0, 1)

def _questions(a):
    qs = a.args.get('questions') or a.args.get('question') or []
    return [qs] if isinstance(qs, str) else [str(q) for q in qs]

def _current(acts): return next((a.summary for a in reversed(acts) if not a.done), '')

def _links(delegates, runs):
    "`{run id: delegate call id}` for each one-question delegate call, matched to the run asked that question."
    out, free = {}, list(runs)
    for a in delegates:
        qs = _questions(a)
        for q in qs:
            if (r := next((r for r in free if r.question.startswith(q.strip())), None)) is None: continue
            free.remove(r)
            if len(qs) == 1: out[r.id] = a.id
    return out

def now_snapshot(agent):
    "What the foreground turn and the sub-agents it started are doing, as a JSON-able dict."
    now, root, acts = time.time(), agent.run(), agent.activity.since()
    ids = {a.id for a in agent.activity.acts}
    mine = [a for a in acts if not a.parent_action_id or a.parent_action_id not in ids]
    runs = [c for c in (root.children if root else []) if c.kind == 'child']
    links = _links([a for a in mine if a.tool in DELEGATE_TOOLS], runs)
    def sub(r):
        calls = [a for a in acts if links.get(r.id) and a.parent_action_id == links[r.id]]
        return {'id': r.id, 'question': _1(r.question, 200), 'elapsed': _secs(r, now),
                'current': _current(calls), 'state': r.state, 'steps': len(calls)}
    return {'at': now, 'busy': agent.busy,
            'root': {'turn_elapsed': _secs(root, now) if root else 0, 'steps': len(mine),
                     'current': _current(mine), 'state': root.state if root else 'idle'},
            'subs': [sub(r) for r in runs]}

A turn that delegates to a sub-agent allowed to write, whose calls are recorded: the scripted model reads a file, then asks one question. Its sub-agent reads the file too and takes a snapshot before it answers, the moment the pane would show both working.

In [ ]:
from ramabana.testing import ScriptedBackend, Step, fake_agent

READ = ('view_file', {'path': '/proj/a.py'})
a, _ = fake_agent()
a.subagent_writes = True
seen = []
class Sub(ScriptedBackend):
    def _run(self, msg):
        yield from super()._run(msg)
        seen.append(now_snapshot(a))
class Root(ScriptedBackend):
    def spawn(self, sp='', tools=(), **kw):
        return Sub(self.spec, steps=[Step(tool=READ), Step('a() is defined')], token_delay=0, tools=tools)
be = Root(steps=[Step(tool=READ), Step(tool=('delegate_search', {'questions': ['what does a.py define?']})), Step('done')],
          token_delay=0, tools=a.tools, sp=a.system_prompt())
a._be = a._be_or_none = lambda job='turn': be
list(a.stream('go'))
mid = seen[0]
test_eq((mid['busy'], mid['root']['steps'], [(s['state'], s['steps']) for s in mid['subs']]), (True, 2, [('running', 1)]))
mid

Once the turn ends the sub-agent is still listed, finished, and the next turn clears it.

In [ ]:
test_eq([s['state'] for s in now_snapshot(a)['subs']], ['completed'])
be.steps = [Step('ok')]
list(a.stream('again'))
test_eq(now_snapshot(a)['subs'], [])

`write_snapshot` replaces the file in one step, so a reader never sees half of it. `read_snapshot` is `None` for a file that is missing or not yet JSON.

In [ ]:
#| export
def write_snapshot(agent, path):
    "Write `now_snapshot(agent)` to `path` atomically."
    p = Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    tmp = p.with_name(f'.{p.name}.{os.getpid()}.tmp')
    tmp.write_text(json.dumps(now_snapshot(agent)))
    os.replace(tmp, p)

def read_snapshot(path):
    "The snapshot at `path`, or `None` when it is missing or unreadable."
    try: return json.loads(Path(path).read_text())
    except (OSError, ValueError): return None

In [ ]:
import tempfile
with tempfile.TemporaryDirectory() as d:
    p = Path(d)/'now.json'
    test_eq(read_snapshot(p), None)
    write_snapshot(a, p)
    test_eq((read_snapshot(p)['busy'], [f.name for f in Path(d).iterdir()]), (False, ['now.json']))
    p.write_text('{"busy": tr')
    test_eq(read_snapshot(p), None)

## Drawing it

`render` draws a snapshot for a pane `width` cells wide: `main` with its steps, clock and current call, then one section per sub-agent with its id, question, clock, current call and a mark, `▶` running, `✓` completed, `✗` stopped or failed. A sub-agent with no recorded calls shows no step count. Lines are cut to the width rather than wrapped. A running clock is advanced by the time since the snapshot was written, so a long call does not look frozen between writes. The colours are the CLI's default `github-dark` palette; `ramabana.cli` is not imported, because it will import this module.

In [ ]:
#| export
PALETTE = {'fg0': '#e6edf3', 'fg1': '#c9d1d9', 'gray': '#8b949e', 'red': '#ff7b72', 'green': '#3fb950',
           'yellow': '#d29922', 'blue': '#58a6ff'}
_STOPPED = ('cancelled', 'detached', 'terminated', 'failed')

def _mark(state):
    if state == 'completed': return '✓', PALETTE['green']
    return ('✗', PALETTE['red']) if state in _STOPPED else ('▶', PALETTE['yellow'])

def _line(s, style, width, indent=''):
    t = Text(indent + s, style=style)
    t.truncate(width, overflow='ellipsis')
    return t

def _clock(secs, state, drift): return f'{secs + (drift if state == "running" else 0):.0f}s'
def _steps(n): return f'{n} step' + ('' if n == 1 else 's')

def render(snap, width=40, now=None):
    "The pane for `snap` as a rich renderable, `width` cells wide."
    if not snap: return _line('waiting for the agent…', PALETTE['gray'], width)
    drift, root = max(0, (now or time.time()) - snap['at']), snap['root']
    bits = [f"step {root['steps']}" if root['steps'] else 'thinking', _clock(root['turn_elapsed'], root['state'], drift)]
    head = Text('main', style=f"bold {PALETTE['blue']}")
    head.append('  ' + (' · '.join(bits) if snap['busy'] else f"idle · {_steps(root['steps'])}"), style=PALETTE['gray'])
    head.truncate(width, overflow='ellipsis')
    rows = [head]
    if snap['busy'] and root['current']: rows.append(_line(root['current'], PALETTE['yellow'], width, '  ⏳ '))
    for s in snap['subs']:
        mark, style = _mark(s['state'])
        t = Text(f"{mark} {s['id']}", style=f'bold {style}')
        t.append(f"  {_clock(s['elapsed'], s['state'], drift)}" + (f" · {_steps(s['steps'])}" if s['steps'] else ''), style=PALETTE['gray'])
        t.truncate(width, overflow='ellipsis')
        rows += [Text(''), t, _line(s['question'], PALETTE['fg1'], width, '  ')]
        if s['current']: rows.append(_line(s['current'], PALETTE['yellow'], width, '  ⏳ '))
    return Group(*rows)

A busy snapshot written five seconds ago, with one sub-agent running, one done and one failed:

In [ ]:
def sub(id, state, secs, current=''):
    return {'id': id, 'question': 'which files import fastllm and why?', 'elapsed': secs, 'current': current,
            'state': state, 'steps': 3}
now = time.time()
busy = {'at': now - 5, 'busy': True,
        'root': {'turn_elapsed': 12, 'steps': 3, 'current': 'Delegate 2 questions', 'state': 'running'},
        'subs': [sub('run_aaa', 'running', 4, 'Search code: fastllm'), sub('run_bbb', 'completed', 2), sub('run_ccc', 'failed', 1)]}
con = Console(record=True, width=40, color_system=None)
con.print(render(busy, 40, now=now))
out = con.export_text()
assert '17s' in out and '▶ run_aaa' in out and '✓ run_bbb' in out and '✗ run_ccc' in out
assert all(len(l) <= 40 for l in out.splitlines())

## The viewer

`ramabana-pane PATH` redraws the file every `every` seconds until ctrl+c. A missing or unreadable file keeps the last good picture on screen.

In [ ]:
#| export
@call_parse
def main(path:str,        # the snapshot file the CLI writes
         every:float=0.2, # seconds between reads
        ):
    "Show the `now` snapshot at `path`, redrawn until ctrl+c."
    con, snap = Console(), None
    try:
        with Live(console=con, auto_refresh=False, transient=True) as live:
            while True:
                snap = read_snapshot(path) or snap
                live.update(render(snap, con.width), refresh=True)
                time.sleep(every)
    except KeyboardInterrupt: pass

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()